Social Value Lancashire

Importing Libraries Below


In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from scipy.stats import spearmanr, linregress
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.impute import SimpleImputer 
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    


Settings Section

In [9]:
EXCEL_FILE = "SocialValueData.xlsx"
CSV_DIR = "results_csv"
FIG_DIR = "figures"

os.makedirs(CSV_DIR, exist_ok=True)

# The 14 real Lancashire local authorities
CANONICAL_LAS = {
    "Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde",
    "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley",
    "Rossendale", "South Ribble", "West Lancashire", "Wyre",
}

#min years and LAs for forecast
MIN_LAS_FOR_FORECAST = 10
MIN_YEARS_FOR_FORECAST = 5
FORECAST_YEARS_AHEAD = 3

# IMD groups to us as predictors of crime
NON_CRIME_IMD_DOMAINS = ["Income", "Employment", "Health", "Education", "Barriers", "Living"]

CLUSTER_COLORS = ["#2A10BE", "#C715A0", "#1BD415", "#DB7413"]

#condensed list features for the correlation heatmap- too many illegible
HEATMAP_FEATURES = [
    "children_low_income_relative_level",
    "residual_waste_per_household_kg_level",
    "pct_waste_recycled_level",
    "anxiety_high_pct_level",
    "life_satisfaction_low_pct_level",
    "rough_sleeping_single_night_level",
    "post16_18_positive_destination_pct_level",
    "pct_adults_active_raw_level",
    "mean_imd_score",
    "imd_domain_Barriers",
    "imd_domain_Health",
    "imd_domain_Living",
]

HEATMAP_LABELS = {
    "gdhi_level": "GDHI (level)", "gdhi_trend": "GDHI (trend)",
    "children_low_income_relative_level": "Child. low income\n(level)",
    "children_low_income_relative_trend": "Child. low income\n(trend)",
    "residual_waste_per_household_kg_level": "Residual waste\n(level)",
    "residual_waste_per_household_kg_trend": "Residual waste\n(trend)",
    "pct_waste_recycled_level": "% waste\nrecycled (level)",
    "pct_waste_recycled_trend": "% waste\nrecycled (trend)",
    "anxiety_high_pct_level": "Anxiety, high\n(level)",
    "anxiety_high_pct_trend": "Anxiety, high\n(trend)",
    "life_satisfaction_low_pct_level": "Life satisf., low\n(level)",
    "life_satisfaction_low_pct_trend": "Life satisf., low\n(trend)",
    "rough_sleeping_single_night_level": "Rough sleeping,\nsingle night (level)",
    "rough_sleeping_single_night_trend": "Rough sleeping,\nsingle night (trend)",
    "post16_18_positive_destination_pct_level": "Post-16-18 positive\ndestination (level)",
    "post16_18_positive_destination_pct_trend": "Post-16-18 positive\ndestination (trend)",
    "pct_adults_active_raw_level": "% adults active,\nraw (level)",
    "pct_adults_active_raw_trend": "% adults active,\nraw (trend)",
    "mean_imd_score": "IMD (overall)",
    "imd_domain_Barriers": "IMD: Barriers",
    "imd_domain_Health": "IMD: Health",
    "imd_domain_Living": "IMD: Living Env.",
}

#Figure stuff
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})

Helper Code

In [10]:
def clean_la_name(name):
    #Cleaning local authority names so consistent
    if pd.isna(name):
        return np.nan
    name = str(name).strip()
    for suffix in [" Borough Council", " City Council", " District Council", " Council", " LA", " District"]:
        if name.endswith(suffix):
            name = name[: -len(suffix)].strip()
    return name


def load_sheet(xls, keyword):
    #find sheet
    matches = [s for s in xls.sheet_names if keyword.lower() in s.lower().replace(" ", "")]
    if not matches:
        raise ValueError(f"No sheet matching '{keyword}' found in the workbook.")
    print(f"  loading sheet '{matches[0]}'")
    return pd.read_excel(xls, sheet_name=matches[0])


1. Loading Workbook

In [11]:
print(f"Looking for {EXCEL_FILE} in {os.getcwd()}")
if not os.path.exists(EXCEL_FILE):
    raise FileNotFoundError(f"'{EXCEL_FILE}' not found, check in the working directory and name correct")
xls = pd.ExcelFile(EXCEL_FILE)
print(f"Opened {EXCEL_FILE}: {len(xls.sheet_names)} sheets\n")

sheets = {
    "income": load_sheet(xls, "GrossDisposableIncome"),
    "children": load_sheet(xls, "ChildrenLowIncome"),
    "waste": load_sheet(xls, "WasteRecycled"),
    "crime": load_sheet(xls, "CrimeRates"),
    "imd": load_sheet(xls, "IMD"),
    "life": load_sheet(xls, "LifeSatisfaction"),
    "rough_sleeping": load_sheet(xls, "SleepingRough"),
    "post16_18": load_sheet(xls, "Post16-18Employment"),
    "active": load_sheet(xls, "AdultsActive"),
}

Looking for SocialValueData.xlsx in c:\Users\Habib\Desktop\SocialValueLancashire
Opened SocialValueData.xlsx: 19 sheets

  loading sheet 'GrossDisposableIncome'
  loading sheet 'ChildrenLowIncome'
  loading sheet '%WasteRecycled'
  loading sheet 'CrimeRates'
  loading sheet 'IMD'
  loading sheet 'LifeSatisfaction'
  loading sheet 'SleepingRough'
  loading sheet 'Post16-18Employment'
  loading sheet '%AdultsActive'


2. Tidy Sheets: Each raw sheet is turned into a tidy (local_authority,year,indicator, value) time series structure

In [12]:
# income
df_income_ts = sheets["income"].copy()
df_income_ts["local_authority"] = df_income_ts["Region name"].apply(clean_la_name)
df_income_ts = df_income_ts[df_income_ts["Transaction"] == "Primary resources total"]
df_income_ts = df_income_ts.rename(columns={"Year": "year", "Value": "value"})
df_income_ts["indicator"] = "gdhi"
df_income_ts = df_income_ts[["local_authority", "year", "indicator", "value"]]

#children in low income families
df_children_ts = sheets["children"].copy()
df_children_ts["local_authority"] = df_children_ts["Local Authority"].apply(clean_la_name)
df_children_ts = df_children_ts.dropna(subset=["local_authority"])
df_children_ts["indicator"] = "children_low_income_" + df_children_ts["Type of Low Income Family"].str.lower()
df_children_ts = df_children_ts.rename(columns={"Year": "year", "Percentage": "value"})
df_children_ts = df_children_ts[["local_authority", "year", "indicator", "value"]]

# waste
df_waste_ts = sheets["waste"].copy()
df_waste_ts["local_authority"] = df_waste_ts["Local Authority"].apply(clean_la_name)
df_waste_ts["Value"] = pd.to_numeric(df_waste_ts["Value"], errors="coerce")
waste_indicator_map = {
    "Percentage of household waste sent for reuse, recycling or composting": "pct_waste_recycled",
    "Residual household waste per household (kg/household)": "residual_waste_per_household_kg",
}
df_waste_ts = df_waste_ts[df_waste_ts["Attribute"].isin(waste_indicator_map)].copy()
df_waste_ts["indicator"] = df_waste_ts["Attribute"].map(waste_indicator_map)
df_waste_ts = df_waste_ts.rename(columns={"Year": "year", "Value": "value"})
df_waste_ts = df_waste_ts[["local_authority", "year", "indicator", "value"]]

# life satisfaction/anxiety
df_life_ts = sheets["life"].copy()
df_life_ts["local_authority"] = df_life_ts["Local Authority"].apply(clean_la_name)
anxiety = df_life_ts[(df_life_ts["ScoreType"] == "Anxiety") & (df_life_ts["Score"] == "High (score 6 to 10) %")].copy()
anxiety["indicator"] = "anxiety_high_pct"
satisfaction = df_life_ts[(df_life_ts["ScoreType"] == "Life Satisfaction") & (df_life_ts["Score"] == "Low (score 0 to 4) %")].copy()
satisfaction["indicator"] = "life_satisfaction_low_pct"
df_life_ts = pd.concat([anxiety, satisfaction]).rename(columns={"Year": "year", "Value": "value"})
df_life_ts = df_life_ts[["local_authority", "year", "indicator", "value"]]

#Monthly counts, averaged up to one value per LA per year per count type
df_rough_ts = sheets["rough_sleeping"].copy()
df_rough_ts["local_authority"] = df_rough_ts["Local Authority"].apply(clean_la_name)
df_rough_ts["year"] = pd.to_datetime(df_rough_ts["Month"]).dt.year
df_rough_ts = (df_rough_ts.groupby(["local_authority", "year", "Time Range"])["Value"]
               .mean().reset_index())
df_rough_ts["indicator"] = df_rough_ts["Time Range"].map({
    "Single Night": "rough_sleeping_single_night",
    "Long Term": "rough_sleeping_longterm",
})
df_rough_ts = df_rough_ts.rename(columns={"Value": "value"})[["local_authority", "year", "indicator", "value"]]

#raw destination counts becomes % going on to apprenticeship or work
df_post1618_ts = sheets["post16_18"].copy()
df_post1618_ts["local_authority"] = df_post1618_ts["Local Authority"].apply(clean_la_name)
df_post1618_ts = df_post1618_ts[df_post1618_ts["Sex"] == "Total"].dropna(subset=["local_authority"])
p1618_wide = df_post1618_ts.pivot_table(index=["local_authority", "Academic Year"], columns="Attribute",
                                          values="Value", aggfunc="sum").reset_index()
for col in ["education", "apprenticeship", "work"]:
    if col not in p1618_wide.columns:
        p1618_wide[col] = 0

positive_destinations = p1618_wide["apprenticeship"] + p1618_wide["work"]
all_destinations = p1618_wide["education"] + positive_destinations
p1618_wide["value"] = positive_destinations / all_destinations * 100
p1618_wide["indicator"] = "post16_18_positive_destination_pct"
df_post1618_ts = p1618_wide.rename(columns={"Academic Year": "year"})[["local_authority", "year", "indicator", "value"]]
df_active_ts = sheets["active"].copy()
df_active_ts["local_authority"] = df_active_ts["Local Authority"].apply(clean_la_name)
df_active_ts = df_active_ts.dropna(subset=["local_authority"])
df_active_ts["year"] = df_active_ts["Date Range"].str.extract(r"(\d{2})$")[0].astype(float) + 2000
df_active_ts["indicator"] = "pct_adults_active_raw"
df_active_ts = df_active_ts.rename(columns={"Value": "value"})[["local_authority", "year", "indicator", "value"]]

#create 1 long table of tidied indicators
panel_long = pd.concat([
    df_income_ts, df_children_ts, df_waste_ts, df_life_ts, df_rough_ts, df_post1618_ts, df_active_ts,
], ignore_index=True)

panel_long = panel_long.dropna(subset=["local_authority"])

#drops rows that arent lancashire districts
panel_dropped = sorted(set(panel_long["local_authority"]) - CANONICAL_LAS)
if panel_dropped:
    print(f"  [filter - time series panel] dropping non-LA rows: {panel_dropped}")
panel_long = panel_long[panel_long["local_authority"].isin(CANONICAL_LAS)]

# force numberic dtypes
before = panel_long["value"].notna().sum()
panel_long["value"] = pd.to_numeric(panel_long["value"], errors="coerce")
after = panel_long["value"].notna().sum()
if after < before:
    print(f"  [numeric coercion] {before - after} values were non-numeric and became NaN")


  [filter - time series panel] dropping non-LA rows: ['Lancashire', 'Lancashire CC', 'Lancashire County', 'North West', 'United Kingdom']
  [numeric coercion] 24 values were non-numeric and became NaN


3. Cross-Sectional Sheets

In [13]:
#Crime is 1 yr only so it's the prediction target, not time series
df_crime = sheets["crime"].copy()
df_crime["local_authority"] = df_crime["Local Authority"].apply(clean_la_name)
df_crime = df_crime[df_crime["Attribute"].str.contains("Total recorded crime", na=False)]
df_crime = df_crime[["local_authority", "Value"]].rename(columns={"Value": "total_recorded_crime"})
df_crime["total_recorded_crime"] = pd.to_numeric(df_crime["total_recorded_crime"], errors="coerce")

#builds the overall IMD score by filtering on 'Attribute' to prevent 
#cross-scale averaging and excluding the Crime to avoid target leakage since redundant.
df_imd = sheets["imd"].copy()
df_imd["local_authority"] = df_imd["Local Authority District name (2024)"].apply(clean_la_name)
imd_scores = df_imd[df_imd["Deprivation Measures"] == "Average score"]

imd_overall = (imd_scores[imd_scores["Attribute"] == "IMD"]
               .groupby("local_authority")["Value"].mean()
               .reset_index().rename(columns={"Value": "mean_imd_score"}))

imd_domains = (imd_scores[imd_scores["Attribute"].isin(NON_CRIME_IMD_DOMAINS)]
               .pivot_table(index="local_authority", columns="Attribute", values="Value", aggfunc="mean")
               .add_prefix("imd_domain_").reset_index())

df_imd = imd_overall.merge(imd_domains, on="local_authority", how="outer")

4. Trajectory features (level/trend per indicator)

In [14]:
#Fits a straight line over time for each LA and indicator to extract the mean level and annual trend slope
#transforming disconnected annual snapshots into a true time-series analysis
indicators = panel_long["indicator"].unique().tolist()
print(f"\nFitting trend lines for indicators: {indicators}")

traj_rows = []
for la, la_group in panel_long.groupby("local_authority"):
    row = {"local_authority": la}
    for indicator in indicators:
        points = la_group[la_group["indicator"] == indicator][["year", "value"]].dropna()
        if points.empty:
            row[f"{indicator}_level"] = np.nan
            row[f"{indicator}_trend"] = np.nan
            continue

        row[f"{indicator}_level"] = float(points["value"].mean())
        if points["year"].nunique() < 2:
            row[f"{indicator}_trend"] = 0.0
        else:
            slope, *_ = linregress(points["year"], points["value"])
            row[f"{indicator}_trend"] = float(slope)
    traj_rows.append(row)

trajectory_df = pd.DataFrame(traj_rows)

n_series = panel_long.groupby(["local_authority", "indicator"]).ngroups
n_possible = panel_long["local_authority"].nunique() * len(indicators)
print(f"Got trend/level features for {n_series}/{n_possible} possible LA x indicator combinations "
      f"(gaps mean that indicator isn't published for that LA in this workbook).")



Fitting trend lines for indicators: ['gdhi', 'children_low_income_relative', 'children_low_income_absolute', 'residual_waste_per_household_kg', 'pct_waste_recycled', 'anxiety_high_pct', 'life_satisfaction_low_pct', 'rough_sleeping_single_night', 'rough_sleeping_longterm', 'post16_18_positive_destination_pct', 'pct_adults_active_raw']
Got trend/level features for 121/154 possible LA x indicator combinations (gaps mean that indicator isn't published for that LA in this workbook).


5. Forecasting

In [15]:
#only indicators with enough local auth and year coverage to trust a trend line forecasted
coverage = (panel_long.dropna(subset=["value"])
            .groupby("indicator")
            .agg(n_las=("local_authority", "nunique"), n_years=("year", "nunique"))
            .reset_index())

forecast_indicators = coverage.loc[
    (coverage["n_las"] >= MIN_LAS_FOR_FORECAST) & (coverage["n_years"] >= MIN_YEARS_FOR_FORECAST),
    "indicator",
].tolist()

excluded = sorted(set(coverage["indicator"]) - set(forecast_indicators))
print(f"\nForecasting {forecast_indicators} (>= {MIN_LAS_FOR_FORECAST} LAs, >= {MIN_YEARS_FOR_FORECAST} years)")
if excluded:
    print(f"Skipping (not enough coverage): {excluded}")

forecast_rows = []
for indicator in forecast_indicators:
    subset = panel_long[panel_long["indicator"] == indicator]
    for la, la_group in subset.groupby("local_authority"):
        clean = la_group.dropna(subset=["year", "value"])
        if len(clean) < 3:
            continue

        years, values = clean["year"].values, clean["value"].values

       #Forecasts N years ahead using a plain linear trend with 
       # +-1.96 SD interval, choosing a simple model over ARIMA 
       # or exponential smoothing because the limited 7-15 annual observations per LA
       #  are insufficient to reliably fit complex autoregressive structures.
        slope, intercept, *_ = linregress(years, values)
        fitted = intercept + slope * years
        resid_std = np.std(values - fitted)
        last_year = int(years.max())

        for h in range(1, FORECAST_YEARS_AHEAD + 1):
            forecast_year = last_year + h
            point = intercept + slope * forecast_year
            forecast_rows.append({
                "indicator": indicator, "local_authority": la, "year": forecast_year,
                "forecast": point, "lower_95": point - 1.96 * resid_std,
                "upper_95": point + 1.96 * resid_std, "trend_slope": slope,
            })

forecast_df = pd.DataFrame(forecast_rows)
forecast_df.to_csv(f"{CSV_DIR}/trend_forecasts.csv", index=False)
print(f"Saved {CSV_DIR}/trend_forecasts.csv ({len(forecast_df)} forecast rows)")


Forecasting ['anxiety_high_pct', 'children_low_income_absolute', 'children_low_income_relative', 'pct_adults_active_raw', 'pct_waste_recycled', 'post16_18_positive_destination_pct', 'residual_waste_per_household_kg', 'rough_sleeping_single_night'] (>= 10 LAs, >= 5 years)
Skipping (not enough coverage): ['gdhi', 'life_satisfaction_low_pct', 'rough_sleeping_longterm']
Saved results_csv/trend_forecasts.csv (330 forecast rows)


6. Assemble base table (one row per Local Auth)

In [19]:
print("\nMerging features into one table per local authority")
abt = trajectory_df.merge(df_crime, on="local_authority", how="outer").merge(df_imd, on="local_authority", how="outer")
ignore_list = ["England", "United Kingdom", "Lancashire Total", "Lancashire", "North West"]
abt = abt.query("local_authority not in @ignore_list")

#drops rows that arent lancashire district
dropped = sorted(set(abt["local_authority"]) - CANONICAL_LAS)
if dropped:
    print(f"[filter - base table] dropping non-LA rows: {dropped}")
abt = abt[abt["local_authority"].isin(CANONICAL_LAS)].reset_index(drop=True)

trajectory_cols = [c for c in trajectory_df.columns if c != "local_authority"]
feature_cols = trajectory_cols + ["mean_imd_score"] + [c for c in abt.columns if c.startswith("imd_domain_")]
target_col = "total_recorded_crime"

#*before* imputing none missing counts printed, so low coverage features eg. GDHI are visible limitation
#rather than just smoothed over by the median imputer
print(f"\nFeature completeness (out of {len(abt)} local authorities):")
for col in feature_cols + [target_col]:
    n_valid = abt[col].notna().sum()
    flag = " *-* low coverage, imputed values will dominate this feature" if n_valid < len(abt) * 0.6 else ""
    print(f"{col:38}: {n_valid}/{len(abt)}{flag}")

#Drops rows with no crime target then median-imputes the remaining features
abt = abt.dropna(subset=[target_col]).reset_index(drop=True)
abt[feature_cols] = SimpleImputer(strategy="median").fit_transform(abt[feature_cols])
print(f"\nFinal table= {len(abt)} local authorities, {len(feature_cols)} features")

#flags feature pairs that are almost duplicates
corr = abt[feature_cols].corr()
high_corr_pairs = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(feature_cols) for b in feature_cols[i + 1:] if abs(corr.loc[a, b]) > 0.85]
if high_corr_pairs:
    print("\nCheck for correlation- highly correlated feature pairs (r > 0.85):")
    for a, b, r in high_corr_pairs:
        print(f"  {a} --- {b}: r={r}")
else:
    print("\nCheck for correlation- No feature pairs exceed r = 0.85.")

corr.to_csv(f"{CSV_DIR}/correlation_matrix_full.csv")
pd.DataFrame(high_corr_pairs, columns=["feature_a", "feature_b", "pearson_r"]).to_csv(
    f"{CSV_DIR}/correlation_high_pairs.csv", index=False)



Merging features into one table per local authority
[filter - base table] dropping non-LA rows: ['Unassigned Lancashire']

Feature completeness (out of 14 local authorities):
gdhi_level                            : 3/14 *-* low coverage, imputed values will dominate this feature
gdhi_trend                            : 3/14 *-* low coverage, imputed values will dominate this feature
children_low_income_relative_level    : 14/14
children_low_income_relative_trend    : 14/14
children_low_income_absolute_level    : 14/14
children_low_income_absolute_trend    : 14/14
residual_waste_per_household_kg_level : 13/14
residual_waste_per_household_kg_trend : 13/14
pct_waste_recycled_level              : 13/14
pct_waste_recycled_trend              : 13/14
anxiety_high_pct_level                : 14/14
anxiety_high_pct_trend                : 14/14
life_satisfaction_low_pct_level       : 7/14 *-* low coverage, imputed values will dominate this feature
life_satisfaction_low_pct_trend       : 7/14 *-* 

7. Clustering

In [ ]:
def fit_cluster_labels(X, algorithm, k):
    if algorithm == "KMeans":
        return KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X)
    if algorithm == "Agglomerative":
        return AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)
    return GaussianMixture(n_components=k, random_state=42).fit(X).predict(X)

X_scaled = StandardScaler().fit_transform(abt[feature_cols])
k_range = range(2, min(6, len(abt) - 1))

#attempt KMeans/Agglomerative/GMM across a range of k and scores each by silhouette rather than arbitrary KMeans(k=3)
results = []
for k in k_range:
    for algorithm in ["KMeans", "Agglomerative", "GMM"]:
        labels = fit_cluster_labels(X_scaled, algorithm, k)
        results.append({"algorithm": algorithm, "k": k, "silhouette": round(silhouette_score(X_scaled, labels), 3)})

comparison_df = pd.DataFrame(results)
print("\nAlgorithm for different k values (silhouette score):")
print(comparison_df.pivot(index="k", columns="algorithm", values="silhouette"))
comparison_df.to_csv(f"{CSV_DIR}/algorithm_comparison.csv", index=False)

best = comparison_df.loc[comparison_df["silhouette"].idxmax()]
best_algo, best_k = best["algorithm"], int(best["k"])
print(f"\nSelected: {best_algo} at k={best_k} (silhouette={best['silhouette']})")

#Resamples with replacement+ refits and compares each resamples labels to the full-sample solution via the Adjusted Rand Index to report 
#cluster stability as a distribution rather than a single run
n = X_scaled.shape[0]
full_labels = fit_cluster_labels(X_scaled, best_algo, best_k)
rng = np.random.RandomState(42)
aris = []
for j in range(200):
    idx = rng.choice(n, size=n, replace=True)
    resample_labels = fit_cluster_labels(X_scaled[idx], best_algo, best_k)
    aris.append(adjusted_rand_score(full_labels[idx], resample_labels))

#measuring and saving how stable and reliable a clustering model is
print(f"\nBootstrap stability for 200 resamples: mean ARI={np.mean(aris):.3f}, sd={np.std(aris):.3f}")
pd.DataFrame({"bootstrap_iteration": range(200), "adjusted_rand_index": aris}).to_csv(
    f"{CSV_DIR}/bootstrap_ari_values.csv", index=False)

abt["la_cluster"] = full_labels
for i in range(best_k):
    group = abt[abt["la_cluster"] == i]
    print(f"\nCluster {i} (size: {len(group)})")
    print(group[["local_authority", "gdhi_level", "mean_imd_score", "total_recorded_crime"]].to_string(index=False))

# Sanity check- does the clustering actually separate high and low-crime areas, or is it just other noise
print("\n**DOES THE CLUSTERING TRACK CRIME LEVELS?**")
cluster_crime_means = abt.groupby("la_cluster")[target_col].mean().sort_values()
print(cluster_crime_means)

# if rho strong +ve (1) crime rate increases reliable and oppisite for -ve (-1)
# if p<0.05 then correlation is statistically significant
rho, p = spearmanr(abt["la_cluster"], abt[target_col])
print(f"Spearman correlation (cluster label vs crime rate): rho={rho:.3f}, p={p:.4f}")


abt.to_csv(f"{CSV_DIR}/results_full_by_local_authority.csv", index=False)
print(f"\nSaved {CSV_DIR}/results_full_by_local_authority.csv ({abt.shape[0]} rows, {abt.shape[1]} columns)")



Algorithm x k comparison (silhouette score):
algorithm  Agglomerative    GMM  KMeans
k                                      
2                  0.294  0.294   0.294
3                  0.262  0.259   0.259
4                  0.253  0.139   0.253
5                  0.188  0.140   0.177

Selected: KMeans at k=2 (silhouette=0.294)

Bootstrap stability for 200 resamples: mean ARI=0.925, sd=0.209

Cluster 0 (size: 6)
      local_authority  gdhi_level  mean_imd_score  total_recorded_crime
Blackburn with Darwen 1803.576923          36.899                 13930
            Blackpool 1917.653846          43.467                 22093
              Burnley 1803.576923          38.681                 10267
             Hyndburn 1803.576923          35.430                  8389
               Pendle 1803.576923          35.285                  6694
              Preston 1803.576923          28.762                 16158

Cluster 1 (size: 8)
local_authority  gdhi_level  mean_imd_score  total_recorded

8. Predictive modelling (Random Forest,Leave-One-Out CV,PCA)

In [ ]:
X_raw = abt[feature_cols].values
y = abt[target_col].values

#Directly compares raw and PCA-reduced performance to test if dimensionality 
# reduction helps, providing statostical evidence since expanding 3 features 
# into 17+ on just 14 rows creates more features than samples.

 #Performs Leave-One-Out Cross-Validation (LOO-CV) to evaluate every LA 
 # as a held-out point exactly once preventing the wild R^2 swings 
 # that a single train/test split would cause given the small sample size 
 # of approximately 14 rows. 
model_raw = RandomForestRegressor(n_estimators=300, random_state=42)
loo_preds_raw = cross_val_predict(model_raw, X_raw, y, cv=LeaveOneOut())
r2_raw = 1 - np.sum((y - loo_preds_raw) ** 2) / np.sum((y - np.mean(y)) ** 2)
print(f"(before pca) LOO-CV R^2 on {X_raw.shape[1]} raw features: {r2_raw:.4f}")
print("--a low or negative score here demonstrates the curse-of-dimensionality problem PCA is meant to fix")

pca_pipe = Pipeline([("scaler", StandardScaler()), ("pca", PCA(n_components=0.85, random_state=42))])
X_reduced = pca_pipe.fit_transform(X_raw)
n_components = pca_pipe.named_steps["pca"].n_components_
explained = pca_pipe.named_steps["pca"].explained_variance_ratio_.sum()
print(f"\n[PCA] Reduced {X_raw.shape[1]} features to {n_components} components ({explained:.1%} of variance)")

model = RandomForestRegressor(n_estimators=300, random_state=42)
loo_preds = cross_val_predict(model, X_reduced, y, cv=LeaveOneOut())
r2 = 1 - np.sum((y - loo_preds) ** 2) / np.sum((y - np.mean(y)) ** 2)
mae = np.mean(np.abs(y - loo_preds))
print(f"(after pca)  LOO-CV MAE: {mae:.2f} crimes, R^2: {r2:.4f}")

model.fit(X_reduced, y)

loo_predictions_df = pd.DataFrame({
    "local_authority": abt["local_authority"].values,
    "actual_total_recorded_crime": y,
    "predicted_raw_features": loo_preds_raw,
    "predicted_pca_features": loo_preds,
})
loo_predictions_df.to_csv(f"{CSV_DIR}/loo_predictions.csv", index=False)
print(f"Saved {CSV_DIR}/loo_predictions.csv ({len(loo_predictions_df)} rows)")

#Projects the model's component importances back onto the original indicators using PCA loadings because need to know which actual indicators matter 
loadings = pca_pipe.named_steps["pca"].components_
component_importance = model.feature_importances_
original_importance = np.abs(loadings.T @ component_importance)
importance_series = pd.Series(original_importance, index=feature_cols).sort_values(ascending=False)

print("\nFeature importance (Random Forest, projected back to original variables):")
for feature, importance in importance_series.items():
    print(f"- {feature:38}: {importance:.4f}")

if HAS_SHAP:
    explainer = shap.TreeExplainer(model)
    shap_values_pc = explainer.shap_values(X_reduced)
    shap_values_original = shap_values_pc @ loadings
    shap_importance = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": np.abs(shap_values_original).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)

    print("\nSHAP feature importance - more robust than raw Random Forest importances:")
    print(shap_importance.to_string(index=False))
    #combine and sort RF and SHAP feature importance scores
    feature_importance_df = (importance_series.rename("rf_importance").rename_axis("feature").reset_index()
                              .merge(shap_importance, on="feature", how="left")
                              .sort_values("rf_importance", ascending=False))
    feature_importance_df.to_csv(f"{CSV_DIR}/feature_importance.csv", index=False)
    print(f"Saved {CSV_DIR}/feature_importance.csv ({len(feature_importance_df)} features, RF + SHAP side by side)")
else:
    shap_importance = None
    print("\n[shap not installed - `pip install shap --break-system-packages` for SHAP-based importance too]")